# Phase 24 Clustering and Dimensionality Reduction Demo

This notebook visualizes deterministic artifacts generated by `backend/scripts/run_phase24_demo_pipeline.py`.

## 1) Global map (100 users)

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path('data')

def load_csv(name: str) -> pd.DataFrame:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing artifact: {path}')
    return pd.read_csv(path)

def load_json(name: str) -> dict:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing artifact: {path}')
    return json.loads(path.read_text(encoding='utf-8'))

In [ ]:
global_before = load_csv('phase24_global_before.csv')
global_after = load_csv('phase24_global_after.csv')

print(f'Global before rows: {len(global_before)}')
print(f'Global after rows: {len(global_after)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(global_before['x'], global_before['y'], s=30, alpha=0.65, label='Before')
ax.scatter(global_after['x'], global_after['y'], s=18, alpha=0.45, label='After')
ax.set_title('Global map for all 100 demo users')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axhline(0.0, color='gray', lw=0.5)
ax.axvline(0.0, color='gray', lw=0.5)
ax.legend()
ax.set_aspect('equal')
plt.show()

## 2) Eleanor Colvin ego subset (20 friends)

In [ ]:
ego_before = load_csv('phase24_eleanor_ego_before.csv')
ego_after = load_csv('phase24_eleanor_ego_after.csv')

print(f'Eleanor ego before rows: {len(ego_before)}')
print(f'Eleanor ego after rows: {len(ego_after)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tiers = ego_before['tier'].astype(int)
scatter = ax.scatter(ego_before['x'], ego_before['y'], c=tiers, cmap='viridis', s=70, alpha=0.8)

for _, row in ego_before.iterrows():
    if row['nickname'] in {'Eleanor Colvin', 'Winston Churchill'}:
        ax.annotate(row['nickname'], (row['x'], row['y']), xytext=(6, 6), textcoords='offset points')

ax.set_title('Eleanor Colvin ego subset before amplification')
ax.set_xlabel('x (Eleanor-centered)')
ax.set_ylabel('y (Eleanor-centered)')
ax.axhline(0.0, color='gray', lw=0.5)
ax.axvline(0.0, color='gray', lw=0.5)
ax.set_aspect('equal')
fig.colorbar(scatter, ax=ax, label='tier')
plt.show()

## 3) Eleanor-centered coordinate shift

In [ ]:
shift = load_csv('phase24_eleanor_shift.csv')

top_shift = shift.assign(distance=((shift['delta_x'] ** 2 + shift['delta_y'] ** 2) ** 0.5))
top_shift = top_shift.sort_values('distance', ascending=False).head(10)
top_shift[['nickname', 'delta_x', 'delta_y', 'distance']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.quiver(
    shift['before_x'],
    shift['before_y'],
    shift['delta_x'],
    shift['delta_y'],
    angles='xy',
    scale_units='xy',
    scale=1,
    alpha=0.55,
)
ax.set_title('Eleanor-centered coordinate shift vectors (before -> after)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axhline(0.0, color='gray', lw=0.5)
ax.axvline(0.0, color='gray', lw=0.5)
ax.set_aspect('equal')
plt.show()

## 4) Side-by-side before/after Eleanor map with boosted Eleanor<->Winston likes/comments

In [ ]:
side_by_side = load_json('phase24_eleanor_side_by_side.json')
before_df = pd.DataFrame(side_by_side['before'])
after_df = pd.DataFrame(side_by_side['after'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, frame, label in [
    (axes[0], before_df, 'Before amplification'),
    (axes[1], after_df, 'After amplification'),
]:
    ax.scatter(frame['x'], frame['y'], c=frame['tier'], cmap='plasma', s=80, alpha=0.85)
    for _, row in frame.iterrows():
        if row['nickname'] in {'Eleanor Colvin', 'Winston Churchill'}:
            ax.annotate(row['nickname'], (row['x'], row['y']), xytext=(6, 6), textcoords='offset points')
    ax.set_title(label)
    ax.axhline(0.0, color='gray', lw=0.5)
    ax.axvline(0.0, color='gray', lw=0.5)
    ax.set_aspect('equal')

fig.suptitle('Eleanor map before/after interaction amplification')
axes[0].set_xlabel('x')
axes[1].set_xlabel('x')
axes[0].set_ylabel('y')
plt.tight_layout()
plt.show()

side_by_side['comparison']